# Hospital Patient Care Analytics Pipeline

This notebook reads the generated CSV file and performs an end-to-end data engineering workflow for the hospital case study.

**Workflow:** Read CSV → Inspect → Clean → Transform → Validate → Analyze → Export

The data is synthetic and intended only for educational use.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
file_path = 'hospital_patient_source_data.csv'
df = pd.read_csv(file_path)
display(df.head())
print('Shape:', df.shape)

## 1. Data Understanding

Inspect column names, data types, missing values, and duplicate records.

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nData types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())
display(df.describe(include='all'))

## 2. Data Cleaning

Remove duplicate records, standardize text fields, and handle missing values.

In [ ]:
df = df.drop_duplicates().copy()

text_columns = ['name', 'gender', 'department', 'doctor_note']
for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

numeric_columns = ['age', 'hemoglobin_result', 'heart_rate', 'steps']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
    df[column] = df[column].fillna(df[column].median())

df['follow_up_required'] = df['follow_up_required'].fillna(False).astype(bool)
print('Rows after cleaning:', len(df))
display(df.head())

## 3. Transformation / Feature Engineering

Calculate waiting time from arrival and consultation times. Create an educational review indicator using simple rule-based conditions.

In [ ]:
df['arrival_datetime'] = pd.to_datetime(df['appointment_date'] + ' ' + df['arrival_time'])
df['consultation_datetime'] = pd.to_datetime(df['appointment_date'] + ' ' + df['consultation_time'])
df['waiting_time_minutes'] = (df['consultation_datetime'] - df['arrival_datetime']).dt.total_seconds() / 60

df['review_indicator'] = (
    (df['age'] >= 65) |
    (df['heart_rate'] >= 100) |
    (df['hemoglobin_result'] < 11) |
    (df['follow_up_required'])
).map({True: 'Review', False: 'Routine'})

display(df[['patient_id', 'waiting_time_minutes', 'review_indicator']])

## 4. Data Validation

Check required columns, valid ages, non-negative waiting times, and allowed review labels.

In [ ]:
required_columns = ['patient_id', 'department', 'age', 'waiting_time_minutes', 'review_indicator']
assert all(column in df.columns for column in required_columns)
assert df['patient_id'].notnull().all()
assert df['patient_id'].is_unique
assert df['age'].between(0, 120).all()
assert (df['waiting_time_minutes'] >= 0).all()
assert df['review_indicator'].isin(['Review', 'Routine']).all()
print('All validation checks passed successfully.')

## 5. Exploratory Data Analysis

Analyze average waiting time, department performance, and review indicators.

In [ ]:
print('Average waiting time:', round(df['waiting_time_minutes'].mean(), 2), 'minutes')
print('Maximum waiting time:', df['waiting_time_minutes'].max(), 'minutes')

department_waiting = (df.groupby('department')['waiting_time_minutes']
                      .mean().round(2)
                      .reset_index(name='average_waiting_time_minutes'))
display(department_waiting)

print('Review indicator counts:')
display(df['review_indicator'].value_counts().reset_index())

In [ ]:
plt.figure(figsize=(8, 4))
department_waiting.plot(kind='bar', x='department', y='average_waiting_time_minutes', legend=False)
plt.title('Average Waiting Time by Department')
plt.xlabel('Department')
plt.ylabel('Average Waiting Time (minutes)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Load the Processed Dataset

Save the cleaned and transformed data as an analytics CSV file.

In [ ]:
final_columns = [
    'patient_id', 'name', 'age', 'gender', 'department', 'appointment_id',
    'appointment_date', 'arrival_time', 'consultation_time',
    'hemoglobin_result', 'heart_rate', 'steps', 'doctor_note',
    'follow_up_required', 'waiting_time_minutes', 'review_indicator'
]

analytics_df = df[final_columns].copy()
output_file = 'hospital_patient_analytics_cleaned.csv'
analytics_df.to_csv(output_file, index=False)
print('Saved:', output_file)
display(analytics_df.head())

## 7. Conclusion

The notebook applies CSV handling, Pandas, data cleaning, transformation, validation, feature engineering, visualization, and loading concepts to the hospital case study.

The final dataset can support operational analysis such as waiting-time monitoring and record review. The review indicator is educational and must not be treated as a clinical diagnosis or validated medical prediction.